# Data Preparation: RoBERTa with Balanced Data 

**Key Features**:
1. Load balanced training data from CSV (original + augmented)
2. Create train/validation/test splits from original data only
3. Add augmented data ONLY to training set (prevents data leakage)
4. RoBERTa tokenization with optimal max_length analysis
5. Save tokenized datasets for training

**Data Source**: `data/balanced_training_data.csv` with leakage prevention

In [1]:
# DuckDB no longer needed - using CSV files
import pandas as pd
import numpy as np
from transformers import RobertaTokenizer
from datasets import Dataset, DatasetDict
from sklearn.preprocessing import LabelEncoder
import pickle
import warnings
warnings.filterwarnings("ignore")

print("✅ Libraries loaded")

✅ Libraries loaded


## 1. Load Data from Balanced CSV (Prevent Data Leakage)

In [2]:
# Load balanced training data from CSV
# Use keep_default_na=False to prevent pandas from converting "None" string to NaN
df = pd.read_csv('data/balanced_training_data.csv', keep_default_na=False)

# Convert empty strings to NaN (actual missing values)
if 'frame_label' in df.columns:
    df['frame_label'] = df['frame_label'].replace('', np.nan)
    # Also handle any other common missing value representations
    df['frame_label'] = df['frame_label'].replace(['nan', 'NaN', 'null', 'NULL'], np.nan)

# Handle "None" as valid label - ensure it's not treated as null
print("⚠️  NOTE: 'None' is treated as a valid frame label, not as null/NaN")

# Remove samples with missing labels
df = df[df['frame_label'].notna()].copy()

# Ensure we have the required columns
if 'chunk_text' not in df.columns:
    raise ValueError("'chunk_text' column not found in dataset")
if 'Annotation' not in df.columns:
    raise ValueError("'Annotation' column not found - needed to separate original from augmented data")

# ============================================================================
# CRITICAL: Prevent data leakage - test/validation sets must ONLY contain original data
# Augmented data should NEVER be in the test or validation sets
# ============================================================================

# Separate original (händisch) and augmented data
original_data = df[df['Annotation'] == 'händisch'].copy()
augmented_data = df[df['Annotation'] == 'augmented'].copy()

print(f"\n📊 Data composition:")
print(f"   Original (händisch): {len(original_data):,} samples")
print(f"   Augmented:          {len(augmented_data):,} samples")
print(f"   Total:               {len(df):,} samples")

# Create train/validation/test splits ONLY from original data
# This ensures test/validation sets contain only unseen, original data
from sklearn.model_selection import train_test_split

print("\n📊 Creating train/validation/test splits from ORIGINAL data only...")
print("   ⚠️  Test and validation sets will contain ONLY original (händisch) data")
print("   ✅ Augmented data will be added ONLY to training set")

# First split: 70% train, 30% temp (which will become val+test)
train_original, temp_df = train_test_split(
    original_data,
    test_size=0.3,
    random_state=42,
    stratify=original_data['frame_label']
)

# Second split: Split temp into 50/50 for validation and test (15% each of original)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df['frame_label']
)

# Add ALL augmented data to training set (augmented data never goes to val/test)
train_full = pd.concat([train_original, augmented_data], ignore_index=True)

print("✅ Splits created successfully")
print(f"   Train (original): {len(train_original):,} samples")
print(f"   Train (augmented): {len(augmented_data):,} samples")
print(f"   Train (total): {len(train_full):,} samples")
print(f"   Validation (original only): {len(val_df):,} samples")
print(f"   Test (original only): {len(test_df):,} samples")

# Verify no augmented data leaked into validation or test sets
val_augmented = (val_df['Annotation'] == 'augmented').sum()
test_augmented = (test_df['Annotation'] == 'augmented').sum()
if val_augmented > 0 or test_augmented > 0:
    raise ValueError(f"❌ DATA LEAKAGE DETECTED: {val_augmented} augmented samples in validation, {test_augmented} in test set!")

print("\n" + "="*80)
print("DATA LOADED FROM data/balanced_training_data.csv")
print("="*80)
print(f"Total samples:  {len(df):,}")
print(f"Train samples: {len(train_full):,} ({len(train_original):,} original + {len(augmented_data):,} augmented)")
print(f"Validation:     {len(val_df):,} samples (100% original, 0% augmented)")
print(f"Test:           {len(test_df):,} samples (100% original, 0% augmented)")
print("="*80)
print("\n✅ Data leakage prevention verified:")
print(f"   - Validation: {val_augmented} augmented samples (should be 0)")
print(f"   - Test: {test_augmented} augmented samples (should be 0)")

⚠️  NOTE: 'None' is treated as a valid frame label, not as null/NaN

📊 Data composition:
   Original (händisch): 2,000 samples
   Augmented:          742 samples
   Total:               2,742 samples

📊 Creating train/validation/test splits from ORIGINAL data only...
   ⚠️  Test and validation sets will contain ONLY original (händisch) data
   ✅ Augmented data will be added ONLY to training set
✅ Splits created successfully
   Train (original): 1,400 samples
   Train (augmented): 742 samples
   Train (total): 2,142 samples
   Validation (original only): 300 samples
   Test (original only): 300 samples

DATA LOADED FROM data/balanced_training_data.csv
Total samples:  2,742
Train samples: 2,142 (1,400 original + 742 augmented)
Validation:     300 samples (100% original, 0% augmented)
Test:           300 samples (100% original, 0% augmented)

✅ Data leakage prevention verified:
   - Validation: 0 augmented samples (should be 0)
   - Test: 0 augmented samples (should be 0)


## 2. Create Label Encoder

In [3]:
# Create label encoder from all data (train + val + test)
all_labels = pd.concat([train_full['frame_label'], val_df['frame_label'], test_df['frame_label']])
label_encoder = LabelEncoder()
label_encoder.fit(all_labels)

# Apply encoding to each split
train_full["label"] = label_encoder.transform(train_full["frame_label"])
val_df["label"] = label_encoder.transform(val_df["frame_label"])
test_df["label"] = label_encoder.transform(test_df["frame_label"])

label_mapping = dict(enumerate(label_encoder.classes_))
print("Label Encoding:")
for idx, label in label_mapping.items():
    print(f"  {idx}: {label}")

# Verify 'None' is present and 'uneindeutig' is not
assert 'None' in label_encoder.classes_, "❌ 'None' label missing!"
assert 'uneindeutig' not in label_encoder.classes_, "❌ 'uneindeutig' should be mapped to 'None'!"
print("\n✅ Label mapping verified: 'uneindeutig' → 'None'")

# Save label encoder
with open("data/roberta_label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)
print("✅ Label encoder saved")

Label Encoding:
  0: Conflict
  1: Economic
  2: Human Impact
  3: Moral Value
  4: None
  5: Powerlessness

✅ Label mapping verified: 'uneindeutig' → 'None'
✅ Label encoder saved


## 3. Verify Data Splits & Leakage Prevention

**Note**: Splits are created from original data only, with augmented data added only to training set.

In [4]:
# Rename train_full to train_df for consistency with rest of notebook
train_df = train_full.copy()

print("="*80)
print("DATA SPLITS VERIFICATION")
print("="*80)
print(f"Train:      {len(train_df):,} samples")
print(f"Validation: {len(val_df):,} samples")
print(f"Test:       {len(test_df):,} samples")
print("="*80)

# Verify data leakage prevention
val_augmented = (val_df['Annotation'] == 'augmented').sum() if 'Annotation' in val_df.columns else 0
test_augmented = (test_df['Annotation'] == 'augmented').sum() if 'Annotation' in test_df.columns else 0
train_augmented = (train_df['Annotation'] == 'augmented').sum() if 'Annotation' in train_df.columns else 0

print("\n✅ Data leakage prevention verified:")
print(f"   - Train: {train_augmented:,} augmented samples (augmented data should be in training)")
print(f"   - Validation: {val_augmented} augmented samples (should be 0)")
print(f"   - Test: {test_augmented} augmented samples (should be 0)")

if val_augmented > 0 or test_augmented > 0:
    print(f"\n❌ WARNING: Data leakage detected!")
    print(f"   Validation has {val_augmented} augmented samples")
    print(f"   Test has {test_augmented} augmented samples")
else:
    print(f"\n✅ No data leakage - validation and test sets contain only original data")
    print(f"   This ensures realistic model evaluation on unseen, original data")

DATA SPLITS VERIFICATION
Train:      2,142 samples
Validation: 300 samples
Test:       300 samples

✅ Data leakage prevention verified:
   - Train: 742 augmented samples (augmented data should be in training)
   - Validation: 0 augmented samples (should be 0)
   - Test: 0 augmented samples (should be 0)

✅ No data leakage - validation and test sets contain only original data
   This ensures realistic model evaluation on unseen, original data


In [5]:
# Check label distribution
print("\nLabel distribution:")
print("\nTrain:")
print(train_df.frame_label.value_counts().sort_index())
print("\nValidation:")
print(val_df.frame_label.value_counts().sort_index())
print("\nTest:")
print(test_df.frame_label.value_counts().sort_index())


Label distribution:

Train:
frame_label
Conflict         325
Economic         320
Human Impact     387
Moral Value      383
None             365
Powerlessness    362
Name: count, dtype: int64

Validation:
frame_label
Conflict         66
Economic         68
Human Impact     35
Moral Value      37
None             46
Powerlessness    48
Name: count, dtype: int64

Test:
frame_label
Conflict         66
Economic         69
Human Impact     35
Moral Value      37
None             46
Powerlessness    47
Name: count, dtype: int64


## 4. Initialize RoBERTa Tokenizer

In [6]:
MODEL_NAME = "roberta-base"

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Loaded {MODEL_NAME}")
print(f"   Vocab size: {tokenizer.vocab_size:,}")

✅ Loaded roberta-base
   Vocab size: 50,265


## 5. Analyze Optimal Max Length

In [7]:
# Sample tokenization to determine optimal max_length
sample_texts = train_df.chunk_text.sample(min(500, len(train_df)), random_state=42).tolist()
token_lengths = [len(tokenizer.encode(text, add_special_tokens=True)) for text in sample_texts]

print("Token length statistics (on sample):")
print(f"  Mean: {np.mean(token_lengths):.1f}")
print(f"  Median: {np.median(token_lengths):.1f}")
print(f"  95th percentile: {np.percentile(token_lengths, 95):.1f}")
print(f"  99th percentile: {np.percentile(token_lengths, 99):.1f}")
print(f"  Max: {np.max(token_lengths):.1f}")

# Determine max_length
if np.percentile(token_lengths, 95) <= 512:
    MAX_LENGTH = 512
    print(f"\n✅ Using MAX_LENGTH = {MAX_LENGTH}")
else:
    MAX_LENGTH = 1024
    print(f"\n✅ Using MAX_LENGTH = {MAX_LENGTH}")
    print(f"⚠️ Note: Some texts will be truncated")

pct_covered = (np.array(token_lengths) <= MAX_LENGTH).mean() * 100
print(f"\n✅ MAX_LENGTH={MAX_LENGTH} covers {pct_covered:.1f}% of samples")

Token length statistics (on sample):
  Mean: 153.4
  Median: 152.5
  95th percentile: 262.0
  99th percentile: 295.1
  Max: 340.0

✅ Using MAX_LENGTH = 512

✅ MAX_LENGTH=512 covers 100.0% of samples


## 6. Tokenize Datasets

In [8]:
# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df[['chunk_text', 'label', 'frame_label']])
val_dataset = Dataset.from_pandas(val_df[['chunk_text', 'label', 'frame_label']])
test_dataset = Dataset.from_pandas(test_df[['chunk_text', 'label', 'frame_label']])

dataset_dict = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

print("Dataset created:")
print(dataset_dict)

Dataset created:
DatasetDict({
    train: Dataset({
        features: ['chunk_text', 'label', 'frame_label'],
        num_rows: 2142
    })
    validation: Dataset({
        features: ['chunk_text', 'label', 'frame_label', '__index_level_0__'],
        num_rows: 300
    })
    test: Dataset({
        features: ['chunk_text', 'label', 'frame_label', '__index_level_0__'],
        num_rows: 300
    })
})


In [9]:
def tokenize_function(examples):
    return tokenizer(
        examples['chunk_text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors=None
    )

print("Tokenizing datasets...")
tokenized_dataset = dataset_dict.map(
    tokenize_function,
    batched=True,
    remove_columns=['chunk_text', 'frame_label'],
    desc="Tokenizing"
)

print("\n✅ Tokenization complete:")
print(tokenized_dataset)

Tokenizing datasets...


Tokenizing:   0%|          | 0/2142 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/300 [00:00<?, ? examples/s]


✅ Tokenization complete:
DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 2142
    })
    validation: Dataset({
        features: ['label', '__index_level_0__', 'input_ids', 'attention_mask'],
        num_rows: 300
    })
    test: Dataset({
        features: ['label', '__index_level_0__', 'input_ids', 'attention_mask'],
        num_rows: 300
    })
})


## 7. Verify Tokenization

In [10]:
idx = 0
print(f"Sample {idx}:")
print(f"Original: {train_df.iloc[idx].chunk_text[:150]}...")
print(f"Label: {train_df.iloc[idx].frame_label} (id={train_df.iloc[idx].label})")
print(f"Input IDs shape: {len(tokenized_dataset['train'][idx]['input_ids'])}")
print(f"Decoded: {tokenizer.decode(tokenized_dataset['train'][idx]['input_ids'][:100])}...")

Sample 0:
Original: The Government’s position has remained unchanged while the final details of the contracts have been ironed out. In November, we set out that we expect...
Label: Economic (id=1)
Input IDs shape: 512
Decoded: <s>The Government’s position has remained unchanged while the final details of the contracts have been ironed out. In November, we set out that we expected to conclude the deal in the coming months, and the Secretary of State made it clear that she was minded to proceed with the contract for difference support package for the deal, subject to any change in circumstances. We remain confident that all parties are firmly behind Hinkley Point C and are working hard towards a final investment decision. We have received...


## 8. Save Processed Datasets & Configuration

In [11]:
# Save tokenized datasets
tokenized_dataset.save_to_disk("data/roberta_tokenized_data")
print("✅ Tokenized datasets saved to data/roberta_tokenized_data/")

# Save configuration
config = {
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'num_labels': len(label_encoder.classes_),
    'label_mapping': label_mapping,
    'train_size': len(train_df),
    'val_size': len(val_df),
    'test_size': len(test_df),
    'group_aware_split': True,
    'split_random_state': 42
}

with open("data/roberta_config.pkl", "wb") as f:
    pickle.dump(config, f)

print("✅ Configuration saved to data/roberta_config.pkl")

Saving the dataset (0/1 shards):   0%|          | 0/2142 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

✅ Tokenized datasets saved to data/roberta_tokenized_data/
✅ Configuration saved to data/roberta_config.pkl


## 9. Summary

In [13]:
print("\n" + "="*80)
print("DATA PREPARATION SUMMARY")
print("="*80)
print(f"\n📊 Dataset:")
print(f"   - Train: {config['train_size']:,} samples")
print(f"   - Validation: {config['val_size']:,} samples")
print(f"   - Test: {config['test_size']:,} samples")
print(f"\n🔢 Labels: {config['num_labels']} classes")
print(f"   {list(label_encoder.classes_)}")
print(f"\n🤖 Model:")
print(f"   - Name: {MODEL_NAME}")
print(f"   - Max length: {MAX_LENGTH} (covers {pct_covered:.1f}% of data)")
print(f"   - Vocab size: {tokenizer.vocab_size:,}")
print(f"\n✅ Data leakage prevention:")
print(f"   - Train: Contains both original and augmented data")
print(f"   - Validation: 100% original data (0% augmented)")
print(f"   - Test: 100% original data (0% augmented)")
print(f"   - This ensures realistic model evaluation on unseen, original data")
print(f"\n💾 Files saved:")
print(f"   - data/roberta_tokenized_data/")
print(f"   - data/roberta_config.pkl")
print(f"   - data/roberta_label_encoder.pkl")
print("="*80)


DATA PREPARATION SUMMARY

📊 Dataset:
   - Train: 2,142 samples
   - Validation: 300 samples
   - Test: 300 samples

🔢 Labels: 6 classes
   ['Conflict', 'Economic', 'Human Impact', 'Moral Value', 'None', 'Powerlessness']

🤖 Model:
   - Name: roberta-base
   - Max length: 512 (covers 100.0% of data)
   - Vocab size: 50,265

✅ Data leakage prevention:
   - Train: Contains both original and augmented data
   - Validation: 100% original data (0% augmented)
   - Test: 100% original data (0% augmented)
   - This ensures realistic model evaluation on unseen, original data

💾 Files saved:
   - data/roberta_tokenized_data/
   - data/roberta_config.pkl
   - data/roberta_label_encoder.pkl
